# Convergence per Attack per Algorithm

In [1]:
%load_ext autoreload
import sys
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pandas as pd
import seaborn as sns
from pathlib import Path
from fastnanoid import generate
from datetime import datetime
import copy

In [2]:
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
lib_path = project_root / "lib"
sys.path.insert(0, str(lib_path))        
sys.path.insert(0, str(project_root))   

In [3]:
%autoreload 2
from lib.graph_factory import GraphFactory, add_graph_plot
from lib.proj_const_estimator import ProjConstEstimator
from lib.utils import fetch_dataset, get_alphas, rng, seed, save_to_pkl, load_from_pkl
from lib.classifier import ByzClassifier, load_run, save_run
from lib.system import SystemSimulator
from lib.metrics import MetricsCalculator
from lib.config import BASE_CONF, NUM_NODES

In [4]:
# Helper Functions
fmt = lambda sec : f"{sec // 60}m {sec % 60:.2f}s"

def convert(o):
    if isinstance(o, np.generic):
        return o.item()
    raise TypeError(f'{type(o)} not serializable')

def get_graph_args(num_byz_nodes, tar_dir_het_param):
    target_pi = rng('dir','graphs').dirichlet(np.full(NUM_NODES, tar_dir_het_param))
    grap_args = {
        'ws_k': 6,
        'ws_p': 0.1,
        'rand_reg_deg':10,
        'geom_radius':0.23,
        'er_p':min(1.0, 3.0 * np.log(NUM_NODES) / NUM_NODES),
        'MH_target_pi':target_pi,
    }
    if(num_byz_nodes == 5):
        r_ub = {
            'erdos-renyi': 0.1461,
            'random-regular': 0.1195,
            'watts-strogatz': 0.0711,
            'geometric': 0.0625,
            'complete': 0.3438
        }
        return grap_args, r_ub
        
    elif(num_byz_nodes == 9):
        grap_args['geometric'] = 0.4
        
        r_ub = {
            'erdos-renyi': 0.043,
            'random-regular': 0.0,
            'watts-strogatz': 0.0,
            'geometric': 0.0,
            'complete': 0.0799
        }
        return grap_args, r_ub
    else: raise ValueError(f'No graph parameters for b={num_byz_nodes}')

The code base in `lib/` holds the implementation of RDSGD, Byzantine 
Experiments and their purpose:
1. Default.

In [5]:
%%capture
RUN_DIR = os.path.join(Path().resolve(), "default")
config = copy.deepcopy(BASE_CONF)
config['train']['b'] = 5
config['sys']['b'] = config['train']['b'] 
config['train']['clf_model'] = 'xgb'
config['data_heterogeneity'] = 100
config['reg_param'] = 1

config['tar_dir_het_param'] = 1
b = config['train']['b']
graph_args, r_ub = get_graph_args(b, config['tar_dir_het_param'])
config['graph_args'] = graph_args
config['graph_weights'] = 'MH'

graphs = ['random-regular','watts-strogatz', 'geometric', 'erdos-renyi']
ATKS = ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM']
ALGS = ['RDSGD', 'ORACLE', 'IOS', 'SCC', 'TriMean', 'CooMed']
ALGS_TEST = ['RDSGD', 'ORACLE']
COLORS = dict(zip(ALGS, ['C0', 'C1', 'C2', 'C3', 'C4', 'C5']))
SEEDS = [111,222,333,555,777]

# Notebook Variables
DEBUG = True
plots = [plt.subplots(1,len(ATKS),figsize=(15,3), sharey=True) for g in graphs]
ta_tables = dict()

In [6]:
payload = copy.deepcopy(config)
del payload['graph_args']['MH_target_pi']
with open(os.path.join(RUN_DIR, f'config.json'), 'w') as f:
    json.dump(payload, f, indent=2, default=convert)

In [7]:
global_dataset = fetch_dataset('MNIST')
gf = GraphFactory(config['train']['num_nodes'], b)
proj_const_estimator = ProjConstEstimator(config, global_dataset, gf) 
clf = ByzClassifier(config, global_dataset, gf)
sys_sim = SystemSimulator(config, global_dataset, gf)
mc = MetricsCalculator(config, global_dataset, rng('metrics-calculator'))

In [8]:
%%time
for g_idx, g in enumerate(graphs):
    config['graph_type']=g
    config['rand_dropout'] = min(r_ub[g], 0.2)
    GRAPH_DIR = os.path.join(RUN_DIR, g)
    os.makedirs(GRAPH_DIR, exist_ok=True)
    fig, ax = plots[g_idx]
    if DEBUG: 
        print(f" {g} graph ".center(56, '='))
    else:
        print(f"{g} graph simulation", end='')
        
    t0 = time.perf_counter()
    
    # Generate simulation data for classifier
    pkl_file = os.path.join(RUN_DIR,f'clf_sim_data_{g}_{config['sys']['b']}.pkl.gz')
    if os.path.exists(pkl_file):
        if DEBUG: print("clf simulations loaded from disk")
        conf = load_run(clf, pkl_file)
        proj_const = conf['proj_const']
    else:
        if DEBUG: print("clf simulations", end='')
        t1 = time.perf_counter()
        proj_const_estimator.configure(config, seed('proj-const-estimator'))
        proj_const = proj_const_estimator.estimate()
        clf_sim_data = clf.run_simulations(config, proj_const, seed('byz-clf','sim'), is_printing=False)
        save_run(clf, pkl_file, {'proj_const':proj_const})
        if DEBUG: print(f" took {fmt(time.perf_counter() - t1)}")
    
    clf_est, clf_preproc = clf.fit()
    clf_op_pt = clf.calc_optimal_op_pt()
    clf_metrics = clf.test()
    
    prelim_metrics = dict()
    prelim_metrics['proj_const'] = proj_const
    prelim_metrics.update(clf_op_pt)
    prelim_metrics.update(clf_metrics)
    if DEBUG: print(f"clf performance: val (γ={clf_op_pt['C_fpr']:.3f}, β={clf_op_pt['C_fnr']:.3f}) test (γ={clf_metrics['gamma_C']:.3f}, β={clf_metrics['beta_C']:.3f})")

    # Record Preliminary Metrics per graph
    clf_metrics_file_path = os.path.join(RUN_DIR, f'clf_metrics_{config['sys']['b']}.csv')
    clf_metrics = copy.deepcopy(prelim_metrics)
    clf_metrics.update({'graph':g})
    if not os.path.exists(clf_metrics_file_path):
        df_clf_metrics = pd.DataFrame([clf_metrics]).set_index('graph')
    else:
        df_clf_metrics = pd.read_csv(clf_metrics_file_path).set_index('graph')
        
        new_row = pd.DataFrame([clf_metrics]).set_index('graph')
        if g in df_clf_metrics.index:
            df_clf_metrics.loc[g] = new_row.loc[g]
        else:
            df_clf_metrics = pd.concat([df_clf_metrics, new_row])
    df_clf_metrics.to_csv(clf_metrics_file_path)

    # Run system simulation
    rows = []
    for atk_idx, atk in enumerate(ATKS):
        for s_idx, s in enumerate([111,222]): # TODO change back
            sim_params = {
                'algorithms': ALGS_TEST,      # TODO change back
                'atk_type': atk,
                'threat_model': 'T3',
                'seed': s
            }
            
            params_C = dict()
            params_C['C_fpr'] = clf_metrics['gamma_C']
            params_C['C_fnr'] = clf_metrics['beta_C']
            params_C['C_tau'] = clf_op_pt['C_tau']

            if DEBUG: print(f"simulation ({atk} attack, seed {s})", end='')
            t1 = time.perf_counter()
            sys_sim.init_simulation(config, proj_const, clf_preproc, clf_est, params_C, seed('sys', s), is_printing_logs=False)
            df = sys_sim.simulate(sim_params, dropout=config['rand_dropout'])
            if(s_idx == 0): df.to_csv(os.path.join(GRAPH_DIR,f'sys_sim_df_{g}_{b}_{atk}.csv'))
            if DEBUG:
                if(s_idx == 0): print(f" took {fmt(time.perf_counter() - t1)}", end=' ')
                else: print(f" took {fmt(time.perf_counter() - t1)}")
                

            # Records for final summary df 
            for alg_idx, alg in enumerate(ALGS_TEST):  #TODO change back
                if(alg == 'RDSGD'):
                    # ASSERTION ONLY TRUE FOR UNIFORM MH WEIGHTS!!
                    # assert(df.loc[alg, 'test_acc_pi'].iloc[-1] == df.loc[alg, 'test_acc'].iloc[-1]) 
                    test_acc = df.loc[alg, 'test_acc_pi'].iloc[-1]
                else:
                    test_acc = df.loc[alg, 'test_acc'].iloc[-1]

                # Plot Lyapunov function for that graph for that attack
                if(s_idx == 0):
                    gap = np.pow(df.loc[alg,'opt_gap'],2)
                    conv = df.loc[alg,'C_unif']
                    ax[atk_idx].plot(conv+gap, color=COLORS[alg])
                    ax[atk_idx].set(title=f'Lyapunov Function $V^{{k}}$ ({atk})', yscale='log')
                    if DEBUG:
                        if(alg_idx == 0): print("(")
                        if(alg_idx != len(ALGS_TEST)-1): print(f"{alg}:{test_acc:.3f}",end=', ')   # TODO change back
                        else: print(f"{alg}:{test_acc:.3f})")
                        
                rows.append({
                    'alg':alg,
                    'atk':atk,
                    'seed':s,
                    'test_acc': test_acc,
                })
                
            # Record system simulation metrics per (graph, atk) tuple
            if(s_idx == 0):
                metrics_file_path = os.path.join(RUN_DIR, f'sys_metrics_{config['sys']['b']}.csv')
                rdsgd_consts = mc(sys_sim.get_sim_config(), sys_sim.models, clf_metrics['gamma_C'], clf_metrics['beta_C'], proj_const)
                pi, x_opt, x_pi_opt = rdsgd_consts['pi'], rdsgd_consts['x_opt'], rdsgd_consts['x_pi_opt']
                del rdsgd_consts['pi']
                del rdsgd_consts['x_opt']
                del rdsgd_consts['x_pi_opt']
                metrics_payload = copy.deepcopy(rdsgd_consts)
                metrics_payload.update({'atk':atk, 'graph':g})

                # Update local .csv file
                if not os.path.exists(metrics_file_path):
                    df_metrics = pd.DataFrame([metrics_payload])
                    df_metrics.set_index(['graph','atk'],inplace=True)
                else:
                    df_metrics = pd.read_csv(metrics_file_path)
                    df_metrics.set_index(['graph','atk'],inplace=True)
                    
                    new_row = pd.DataFrame([metrics_payload]).set_index(['graph', 'atk'])
                    if (g, atk) in df_metrics.index:
                        df_metrics.loc[(g, atk)] = new_row.loc[(g, atk)]
                    else:
                        df_metrics = pd.concat([df_metrics, new_row])
                df_metrics.to_csv(metrics_file_path)

    if not DEBUG: print(f" took {fmt(time.perf_counter() - t0)}")
    
    # Record test accuracy final df   
    df_summary = pd.DataFrame(rows).groupby(['atk', 'alg'])['test_acc'].agg(['mean', 'std']).unstack('alg')
    ta_tables[g] = df_summary
    print(f"{g} graph test accuracy table")
    display(df_summary)

================= random-regular graph =================
clf simulations loaded from disk
clf performance: val (γ=0.006, β=0.139) test (γ=0.009, β=0.136)
simulation (label_flip attack, seed 111) took 1.0m 24.90s
RDSGD:0.849, ORACLE:0.852
simulation (label_flip attack, seed 222) took 1.0m 22.03s
simulation (sign_flip attack, seed 111) took 1.0m 13.80s
RDSGD:0.846, ORACLE:0.852
simulation (sign_flip attack, seed 222) took 1.0m 14.70s
simulation (gaussian attack, seed 111) took 1.0m 12.64s
RDSGD:0.851, ORACLE:0.852
simulation (gaussian attack, seed 222) took 1.0m 13.10s
simulation (ALIE attack, seed 111) took 1.0m 12.01s
RDSGD:0.851, ORACLE:0.852
simulation (ALIE attack, seed 222) took 1.0m 13.45s
simulation (IPM attack, seed 111) took 1.0m 11.08s
RDSGD:0.851, ORACLE:0.852
simulation (IPM attack, seed 222) took 1.0m 9.80s
random-regular graph test accuracy table


mean                 std          
alg           ORACLE     RDSGD    ORACLE     RDSGD
atk                                               
ALIE        0.852857  0.852056  0.001102  0.001622
IPM         0.852922  0.852143  0.001010  0.001622
gaussian    0.852879  0.851861  0.001500  0.001347
label_flip  0.853182  0.850476  0.001867  0.002694
sign_flip   0.852554  0.848593  0.000980  0.003030

================= watts-strogatz graph =================
clf simulations took 3.0m 13.15s
clf performance: val (γ=0.007, β=0.185) test (γ=0.006, β=0.159)
simulation (label_flip attack, seed 111) took 1.0m 1.95s
RDSGD:0.851, ORACLE:0.851
simulation (label_flip attack, seed 222) took 1.0m 5.54s
simulation (sign_flip attack, seed 111) took 1.0m 3.42s
RDSGD:0.845, ORACLE:0.852
simulation (sign_flip attack, seed 222) took 1.0m 6.91s
simulation (gaussian attack, seed 111) took 1.0m 8.47s
RDSGD:0.851, ORACLE:0.852
simulation (gaussian attack, seed 222) took 1.0m 2.97s
simulation (ALIE attack, seed 111) took 1.0m 4.31s
RDSGD:0.851, ORACLE:0.852
simulation (ALIE attack, seed 222) took 1.0m 2.76s
simulation (IPM attack, seed 111) took 1.0m 8.17s
RDSGD:0.851, ORACLE:0.852
simulation (IPM attack, seed 222) took 1.0m 21.69s
watts-strogatz graph test accuracy table


mean                 std          
alg           ORACLE     RDSGD    ORACLE     RDSGD
atk                                               
ALIE        0.852489  0.852100  0.000765  0.002051
IPM         0.852511  0.852403  0.000735  0.002051
gaussian    0.852684  0.852035  0.000429  0.001347
label_flip  0.852489  0.851970  0.001806  0.001928
sign_flip   0.852056  0.846970  0.000704  0.003122

=================== geometric graph ====================
clf simulations

...

 took 0.0m 50.66s
geometric graph test accuracy table


mean                 std          
alg           ORACLE     RDSGD    ORACLE     RDSGD
atk                                               
ALIE        0.852922  0.851558  0.001439  0.002020
IPM         0.853052  0.851667  0.001133  0.002173
gaussian    0.852597  0.851970  0.001469  0.001500
label_flip  0.853030  0.851861  0.001041  0.001347
sign_flip   0.852554  0.847338  0.002143  0.004745

================== erdos-renyi graph ===================
clf simulations took 4.0m 8.18s
clf performance: val (γ=0.011, β=0.152) test (γ=0.011, β=0.155)
simulation (label_flip attack, seed 111) took 1.0m 9.58s
RDSGD:0.851, ORACLE:0.852
simulation (label_flip attack, seed 222) took 1.0m 7.44s
simulation (sign_flip attack, seed 111) took 1.0m 7.21s
RDSGD:0.847, ORACLE:0.852
simulation (sign_flip attack, seed 222) took 1.0m 13.12s
simulation (gaussian attack, seed 111) took 1.0m 32.81s
RDSGD:0.851, ORACLE:0.853
simulation (gaussian attack, seed 222) took 1.0m 21.33s
simulation (ALIE attack, seed 111) took 1.0m 13.87s
RDSGD:0.851, ORACLE:0.853
simulation (ALIE attack, seed 222) took 1.0m 8.87s
simulation (IPM attack, seed 111) took 1.0m 7.61s
RDSGD:0.851, ORACLE:0.853
simulation (IPM attack, seed 222) took 1.0m 8.61s
erdos-renyi graph test accuracy table


mean                 std          
alg           ORACLE     RDSGD    ORACLE     RDSGD
atk                                               
ALIE        0.853052  0.852121  0.000214  0.001469
IPM         0.853052  0.852208  0.000643  0.001224
gaussian    0.852965  0.851970  0.000643  0.001500
label_flip  0.852684  0.852035  0.001531  0.002082
sign_flip   0.852446  0.848528  0.001071  0.002020

CPU times: user 16h 16min 50s, sys: 9min 7s, total: 16h 25min 58s
Wall time: 55min 33s
